## Google Colab Setup - Install Dependencies
Install all required packages with specific versions for compatibility

In [ ]:
# Install required packages with latest stable versions
!pip install -q \
    langgraph==0.2.60 \
    langchain==0.3.13 \
    langchain-openai==0.2.14 \
    langchain-community==0.3.13 \
    langchain-experimental==0.3.3 \
    playwright==1.49.0 \
    gradio==5.12.0 \
    pydantic==2.10.4 \
    python-dotenv==1.0.1 \
    nest-asyncio==1.6.0 \
    wikipedia==1.4.0 \
    google-search-results==2.4.2

In [ ]:
# Install Playwright browsers (required for web automation)
!playwright install chromium
print("✓ Playwright browsers installed")

In [ ]:
# Configure Colab event loop for async Gradio
import nest_asyncio
nest_asyncio.apply()
print("✓ Event loop configured for Colab")

## Imports and Setup

In [ ]:
# Core LangGraph and LangChain imports
from typing import Annotated, TypedDict, List, Dict, Any, Optional, Literal
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_async_playwright_browser
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field
from IPython.display import Image, display, Markdown
import gradio as gr
import uuid
import nest_asyncio
import asyncio
import os
import re
import traceback
import requests
from datetime import datetime
from dotenv import load_dotenv

# nest_asyncio already applied in previous cell
print("✓ All imports loaded successfully")

In [ ]:
# Load environment variables from Colab Secrets
# In Colab: Settings → Secrets → Add OPENAI_API_KEY, PUSHOVER_TOKEN, PUSHOVER_USER
load_dotenv(override=True)

# Verify API key is available
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("⚠️ WARNING: OPENAI_API_KEY not found! Please set it in Colab Secrets.")
    print("Go to: Settings → Secrets → Add 'OPENAI_API_KEY' with your key value")
else:
    print("✓ OPENAI_API_KEY loaded successfully")

# Check Pushover credentials (optional)
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_user = os.getenv("PUSHOVER_USER")
if pushover_token and pushover_user:
    print("✓ Pushover credentials loaded - notifications enabled")
else:
    print("ℹ️ Pushover credentials not found - notifications disabled (optional feature)")

## Structured Output Definition
Pydantic models for type-safe structured outputs from the evaluator

In [ ]:
class EvaluatorOutput(BaseModel):
    """
    Structured output schema for the evaluator agent.
    Ensures consistent evaluation responses with clear fields.
    """
    feedback: str = Field(
        description="Detailed feedback on the assistant's response quality, accuracy, and completeness"
    )
    success_criteria_met: bool = Field(
        description="Boolean indicating whether the success criteria have been fully met"
    )
    user_input_needed: bool = Field(
        description="True if more input is needed from user, clarifications required, or assistant is stuck"
    )

print("✓ EvaluatorOutput schema defined")

## State Definition
TypedDict for managing graph state with message reduction

In [ ]:
class State(TypedDict):
    """
    State container for the Sidekick multi-agent system.
    Uses Annotated with add_messages reducer for proper message accumulation.
    """
    messages: Annotated[List[Any], add_messages]
    success_criteria: str
    feedback_on_work: Optional[str]
    success_criteria_met: bool
    user_input_needed: bool
    notification_sent: bool

print("✓ State schema defined")

## Pushover Notification Tool
Send real-time push notifications to mobile devices

In [ ]:
# Pushover API configuration
PUSHOVER_API_URL = "https://api.pushover.net/1/messages.json"

def send_push_notification(message: str, title: str = "Sidekick Update") -> str:
    """
    Sends a push notification via Pushover API.
    
    Args:
        message: The notification message content
        title: Optional title for the notification
        
    Returns:
        str: 'success' if sent, error message otherwise
    """
    try:
        token = os.getenv("PUSHOVER_TOKEN")
        user = os.getenv("PUSHOVER_USER")
        
        if not token or not user:
            return "Pushover credentials not configured"
        
        payload = {
            "token": token,
            "user": user,
            "message": message,
            "title": title,
            "priority": 0,
            "timestamp": int(datetime.now().timestamp())
        }
        
        response = requests.post(PUSHOVER_API_URL, data=payload, timeout=10)
        
        if response.status_code == 200:
            print(f"✓ Push notification sent: {message[:50]}...")
            return "success"
        else:
            error_msg = f"Pushover API error: {response.status_code} - {response.text}"
            print(error_msg)
            return error_msg
            
    except requests.exceptions.Timeout:
        return "Pushover request timed out"
    except requests.exceptions.RequestException as e:
        return f"Pushover request failed: {str(e)}"
    except Exception as e:
        return f"Unexpected error sending notification: {str(e)}"


def test_pushover_notification() -> str:
    """
    Test function to verify Pushover integration is working.
    
    Returns:
        str: Test result message
    """
    result = send_push_notification(
        message="🚀 Sidekick notification system test successful!",
        title="Sidekick Test"
    )
    return f"Notification test result: {result}"

print("✓ Pushover notification functions defined")

## Input/Output Guardrails
Comprehensive validation and sanitization functions

In [ ]:
# ============= INPUT GUARDRAILS =============

def validate_input_text(text: str) -> bool:
    """
    Validates user input text for safety and appropriateness.
    Checks for empty input, excessive length, and potentially harmful patterns.
    
    Args:
        text: The user input text to validate
        
    Returns:
        bool: True if input is valid, False otherwise
    """
    if not text or not text.strip():
        return False  # Reject empty input
    if len(text) > 5000:
        return False  # Prevent excessively long inputs
    # Basic injection attack prevention
    if '\x00' in text:
        return False  # Reject null bytes
    return True


def validate_success_criteria(criteria: str) -> bool:
    """
    Validates success criteria input for clarity and actionability.
    Ensures criteria is specific, measurable, and achievable.
    
    Args:
        criteria: The success criteria string to validate
        
    Returns:
        bool: True if criteria is valid, False otherwise
    """
    if not criteria or not criteria.strip():
        return False  # Reject empty criteria
    if len(criteria) > 2000:
        return False  # Prevent excessively long criteria
    # Check for minimum specificity (at least 10 characters)
    if len(criteria.strip()) < 10:
        return False
    return True


def sanitize_input(text: str) -> str:
    """
    Sanitizes user input by removing potentially problematic characters.
    Prevents injection attacks and ensures clean, normalized input.
    
    Args:
        text: The input text to sanitize
        
    Returns:
        str: Sanitized and normalized text
    """
    # Remove null bytes
    sanitized = text.replace('\x00', '')
    # Normalize whitespace (multiple spaces/tabs/newlines to single space)
    sanitized = re.sub(r'\s+', ' ', sanitized)
    return sanitized.strip()


# ============= OUTPUT GUARDRAILS =============

def validate_output_content(content: str) -> bool:
    """
    Validates AI-generated output for safety and quality.
    Checks for appropriate length and basic content quality.
    
    Args:
        content: The AI-generated content to validate
        
    Returns:
        bool: True if output is valid, False otherwise
    """
    if not content or not content.strip():
        return False  # Reject empty output
    if len(content) > 10000:
        return False  # Prevent excessively long outputs
    return True


def filter_sensitive_info(text: str) -> str:
    """
    Filters out potentially sensitive information from output.
    Uses pattern matching to detect and redact sensitive data patterns.
    
    Args:
        text: The text to filter
        
    Returns:
        str: Filtered text with sensitive info redacted
    """
    # Mask potential API keys (long alphanumeric strings)
    text = re.sub(r'\b[A-Za-z0-9]{32,}\b', '[REDACTED_API_KEY]', text)
    # Mask potential email addresses
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '[EMAIL]', text)
    # Mask potential phone numbers (basic pattern)
    text = re.sub(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', '[PHONE]', text)
    return text


def apply_guardrails_to_message(message: str, message_type: Literal["input", "output"] = "input") -> tuple[bool, str]:
    """
    Applies comprehensive guardrails to a message.
    Combines validation and sanitization/filtering in one function.
    
    Args:
        message: The message to validate and process
        message_type: Either 'input' for user input or 'output' for AI output
        
    Returns:
        tuple: (is_valid, processed_message_or_error)
            - is_valid: Boolean indicating if message passed validation
            - processed_message: Sanitized/filtered message if valid
            - error_message: Error description if invalid
    """
    try:
        if message_type == "input":
            if not validate_input_text(message):
                return False, "Invalid input: Please provide a non-empty message (max 5000 chars)."
            sanitized = sanitize_input(message)
            return True, sanitized
            
        elif message_type == "output":
            if not validate_output_content(message):
                return False, "Invalid output generated: Content validation failed."
            filtered = filter_sensitive_info(message)
            return True, filtered
            
    except Exception as e:
        return False, f"Guardrail processing error: {str(e)}"
    
    return True, message


print("✓ Guardrail functions defined")

## Browser Tools Setup
Initialize Playwright browser toolkit for web automation

In [ ]:
# Initialize async Playwright browser tools (headless for Colab)
async_browser = None
toolkit = None
tools = []

try:
    async_browser = create_async_playwright_browser(headless=True)
    toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=async_browser)
    tools = toolkit.get_tools()
    print(f"✓ Browser tools initialized successfully ({len(tools)} tools available)")
except Exception as e:
    print(f"⚠ Browser tools initialization warning: {e}")
    print("ℹ️ Continuing without browser tools - other tools will still work")

## Additional Tools Setup
Search, Wikipedia, Python REPL, File Management, and Pushover

In [ ]:
from langchain.agents import Tool
from langchain_community.agent_toolkits import FileManagementToolkit
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper
from langchain_experimental.tools import PythonREPLTool
from langchain_community.utilities import GoogleSerperAPIWrapper

# Create Pushover notification tool
pushover_tool = Tool(
    name="send_push_notification",
    func=send_push_notification,
    description="Use this tool to send push notifications to the user's mobile device via Pushover. "
                "Input should be the message you want to send. Use when task completion or important updates should be notified."
)

# File management tools (sandboxed)
file_toolkit = FileManagementToolkit(root_dir="sandbox")
file_tools = file_toolkit.get_tools()

# Web search tool (requires SERPER_API_KEY)
serper = GoogleSerperAPIWrapper()
search_tool = Tool(
    name="web_search",
    func=serper.run,
    description="Use this tool to search the web for current information, news, or specific data. "
                "Input should be a search query."
)

# Wikipedia tool
wikipedia_api = WikipediaAPIWrapper(top_k_results=3, doc_content_chars_max=2000)
wiki_tool = WikipediaQueryRun(api_wrapper=wikipedia_api)

# Python REPL tool for code execution
python_repl = PythonREPLTool()

# Combine all tools
all_tools = tools + file_tools + [pushover_tool, search_tool, python_repl, wiki_tool]
print(f"✓ Total tools available: {len(all_tools)}")
print(f"  - Browser tools: {len(tools)}")
print(f"  - File tools: {len(file_tools)}")
print(f"  - Special tools: Pushover, Search, Python REPL, Wikipedia")

## LLM Initialization with Model Parameters
Configure worker and evaluator LLMs with sampling parameters

In [ ]:
# Worker LLM - optimized for task execution with tools
worker_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,           # Balanced creativity and consistency
    max_tokens=2048,          # Max generated tokens
    streaming=True,           # Enable streaming for real-time responses
    timeout=60,               # Request timeout in seconds
    max_retries=2             # Automatic retry on failures
)

# Bind tools to worker LLM
worker_llm_with_tools = worker_llm.bind_tools(all_tools) if all_tools else worker_llm

# Evaluator LLM - optimized for consistent, reliable evaluation
evaluator_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,           # Lower temperature for consistent evaluation
    max_tokens=1024,          # Sufficient for feedback
    streaming=False,          # No streaming needed for evaluator
    timeout=30,
    max_retries=2
)

# Configure evaluator with structured output
evaluator_llm_with_output = evaluator_llm.with_structured_output(EvaluatorOutput)

print("✓ LLMs initialized with parameters:")
print(f"  Worker: gpt-4o-mini, temp=0.7, max_tokens=2048, streaming=True")
print(f"  Evaluator: gpt-4o-mini, temp=0.3, max_tokens=1024, structured_output=True")

## Worker Node Implementation
The worker agent that performs tasks using tools with exception handling

In [ ]:
def worker(state: State) -> Dict[str, Any]:
    """
    Worker node that processes tasks and uses tools to complete assignments.
    Implements comprehensive exception handling and proper state management.
    
    Args:
        state: Current graph state containing messages, success criteria, and feedback
        
    Returns:
        Dict containing updated messages with the worker's response
    """
    try:
        # Build system message with success criteria and context
        system_message = f"""You are a helpful AI assistant that can use tools to complete tasks.
You keep working on a task until either you have a question or clarification for the user, or the success criteria is met.
You have many tools to help you, including:
- Web browsing and navigation
- Web search for current information
- Wikipedia for factual knowledge
- Python code execution
- File management
- Push notifications via Pushover

This is the success criteria for your current task:
{state['success_criteria']}

You should reply either with a question for the user about this assignment, or with your final response.
If you have a question for the user, you need to reply by clearly stating your question. Example:

Question: please clarify whether you want a summary or a detailed answer

If you've finished, reply with the final answer, and don't ask a question; simply reply with the answer.

Current date and time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
"""
        
        # Add feedback context if this is a retry attempt
        if state.get("feedback_on_work"):
            system_message += f"""

IMPORTANT: Previously you thought you completed the assignment, but your reply was rejected because the success criteria was not met.
Here is the feedback on why this was rejected:
{state['feedback_on_work']}

With this feedback, please continue the assignment, ensuring that you meet the success criteria or have a question for the user.
"""
        
        # Prepare messages list with system message
        messages = list(state["messages"])
        found_system_message = False
        
        # Update existing system message or add new one
        for message in messages:
            if isinstance(message, SystemMessage):
                message.content = system_message
                found_system_message = True
        
        if not found_system_message:
            messages = [SystemMessage(content=system_message)] + messages
        
        # Invoke LLM with tools
        response = worker_llm_with_tools.invoke(messages)
        
        # Apply output guardrails if response has content
        if hasattr(response, 'content') and response.content:
            is_valid, processed_content = apply_guardrails_to_message(response.content, "output")
            if not is_valid:
                response.content = f"Output validation failed: {processed_content}"
            else:
                response.content = processed_content
        
        return {"messages": [response]}
        
    except Exception as e:
        # Comprehensive error handling with full traceback
        error_details = traceback.format_exc()
        error_message = f"""⚠️ Worker Error Occurred

Error: {str(e)}

Traceback:
{error_details[:1000]}..."""
        
        print(f"❌ Worker error: {str(e)}")
        return {"messages": [AIMessage(content=error_message)]}


print("✓ Worker node defined")

## Worker Router
Routes worker output to tools or evaluator based on tool calls

In [ ]:
def worker_router(state: State) -> Literal["tools", "evaluator"]:
    """
    Routes worker output based on whether tool calls are present.
    If the worker wants to use tools, route to ToolNode. Otherwise, route to evaluator.
    
    Args:
        state: Current graph state
        
    Returns:
        str: 'tools' if tool calls present, 'evaluator' otherwise
    """
    try:
        last_message = state["messages"][-1]
        
        # Check if the message contains tool calls
        if hasattr(last_message, "tool_calls") and last_message.tool_calls:
            return "tools"
        else:
            return "evaluator"
            
    except Exception as e:
        print(f"⚠️ Worker router error: {e}")
        return "evaluator"  # Default to evaluator on error for safety


print("✓ Worker router defined")

## Conversation Formatter
Formats conversation history for evaluator consumption

In [ ]:
def format_conversation(messages: List[Any]) -> str:
    """
    Formats conversation history into readable text for the evaluator.
    Handles different message types (Human, AI, System, Tool).
    
    Args:
        messages: List of message objects from the conversation
        
    Returns:
        str: Formatted conversation text with role labels
    """
    conversation = "Conversation history:\n\n"
    
    for message in messages:
        if isinstance(message, HumanMessage):
            conversation += f"User: {message.content}\n"
        elif isinstance(message, AIMessage):
            text = message.content or "[Using tools...]"
            # Truncate very long messages for readability
            if len(text) > 500:
                text = text[:500] + "..."
            conversation += f"Assistant: {text}\n"
        elif isinstance(message, SystemMessage):
            # Only include first 200 chars of system messages
            conversation += f"System: {message.content[:200]}...\n"
        elif isinstance(message, ToolMessage):
            # Truncate tool results
            content = message.content[:200] if len(message.content) > 200 else message.content
            conversation += f"Tool Result: {content}...\n"
    
    return conversation


print("✓ Conversation formatter defined")

## Evaluator Node Implementation
The evaluator agent that assesses worker output quality using structured outputs

In [ ]:
def evaluator(state: State) -> State:
    """
    Evaluator node that assesses worker output quality.
    Uses structured output (Pydantic) for consistent, type-safe evaluation.
    Implements comprehensive exception handling.
    
    Args:
        state: Current graph state with messages and criteria
        
    Returns:
        Updated state with evaluation results and feedback
    """
    try:
        # Get the worker's last response
        last_response = state["messages"][-1].content

        # System message for evaluator role
        system_message = """You are an expert evaluator that determines if a task has been completed successfully by an Assistant.
Assess the Assistant's last response based on the given criteria. Respond with detailed feedback, and with your decision on whether the success criteria has been met,
and whether more input is needed from the user.

Be fair but thorough. If the assistant has made a good effort and the response is reasonable, approve it.
If the assistant says they've written a file or completed an action, assume they have done so unless there's clear evidence otherwise."""
        
        # User message with full context
        user_message = f"""You are evaluating a conversation between the User and Assistant. You decide what action to take based on the last response from the Assistant.

The entire conversation with the assistant, with the user's original request and all replies, is:
{format_conversation(state['messages'])}

The success criteria for this assignment is:
{state['success_criteria']}

And the final response from the Assistant that you are evaluating is:
{last_response}

Respond with your feedback, and decide if the success criteria is met by this response.
Also, decide if more user input is required, either because the assistant has a question, needs clarification, or seems to be stuck and unable to answer without help.

If you're seeing the Assistant repeating the same mistakes, then consider responding that user input is required.
"""
        
        # Add previous feedback context if available
        if state["feedback_on_work"]:
            user_message += f"\nNote: In a prior attempt, you provided this feedback: {state['feedback_on_work']}"
        
        # Prepare evaluator messages
        evaluator_messages = [
            SystemMessage(content=system_message),
            HumanMessage(content=user_message)
        ]

        # Invoke evaluator LLM with structured output
        eval_result = evaluator_llm_with_output.invoke(evaluator_messages)
        
        # Build new state with evaluation results
        new_state = {
            "messages": [{
                "role": "assistant",
                "content": f"📊 **Evaluator Feedback:** {eval_result.feedback}"
            }],
            "feedback_on_work": eval_result.feedback,
            "success_criteria_met": eval_result.success_criteria_met,
            "user_input_needed": eval_result.user_input_needed,
            "notification_sent": False
        }
        
        # Send push notification if task completed
        if eval_result.success_criteria_met:
            try:
                notification_result = send_push_notification(
                    message=f"✅ Task completed! {eval_result.feedback[:100]}...",
                    title="Sidekick Success"
                )
                if notification_result == "success":
                    new_state["notification_sent"] = True
            except Exception as notify_error:
                print(f"Notification error: {notify_error}")
        
        return new_state
        
    except Exception as e:
        # Handle evaluator errors gracefully
        error_details = traceback.format_exc()
        error_feedback = f"""⚠️ Evaluator Error Occurred

Error: {str(e)}

This is a system error. Please try again or report the issue.
Traceback: {error_details[:500]}..."""
        
        print(f"❌ Evaluator error: {str(e)}")
        return {
            "messages": [{"role": "assistant", "content": error_feedback}],
            "feedback_on_work": error_feedback,
            "success_criteria_met": False,
            "user_input_needed": True,
            "notification_sent": False
        }


print("✓ Evaluator node defined")

## Evaluation Router
Routes based on evaluation results to continue or end

In [ ]:
def route_based_on_evaluation(state: State) -> Literal["worker", "END"]:
    """
    Routes flow based on evaluation results.
    If criteria met or user input needed, end the conversation.
    Otherwise, send back to worker for another iteration.
    
    Args:
        state: Current graph state with evaluation results
        
    Returns:
        str: 'worker' to continue working, 'END' to finish
    """
    try:
        # End if success criteria met or user input is needed
        if state["success_criteria_met"] or state["user_input_needed"]:
            return "END"
        else:
            return "worker"
            
    except Exception as e:
        print(f"⚠️ Route evaluation error: {e}")
        return "END"  # Default to end on error for safety


print("✓ Evaluation router defined")

## Graph Construction
Build and compile the LangGraph workflow with all nodes and edges

In [ ]:
# Build the StateGraph
graph_builder = StateGraph(State)

# Add nodes to the graph
graph_builder.add_node("worker", worker)
if all_tools:  # Only add tools node if tools are available
    graph_builder.add_node("tools", ToolNode(tools=all_tools))
graph_builder.add_node("evaluator", evaluator)

# Add conditional edges based on worker output
if all_tools:
    graph_builder.add_conditional_edges(
        "worker",
        worker_router,
        {
            "tools": "tools",      # Route to tools if tool calls present
            "evaluator": "evaluator"  # Route to evaluator otherwise
        }
    )
    # Tools loop back to worker after execution
    graph_builder.add_edge("tools", "worker")
else:
    # No tools available, always route to evaluator
    graph_builder.add_conditional_edges(
        "worker",
        worker_router,
        {"evaluator": "evaluator"}
    )

# Add conditional edge from evaluator based on evaluation
graph_builder.add_conditional_edges(
    "evaluator",
    route_based_on_evaluation,
    {
        "worker": "worker",  # Continue working if criteria not met
        "END": END           # End if success or user input needed
    }
)

# Add start edge (entry point is worker)
graph_builder.add_edge(START, "worker")

# Compile with memory checkpoint for conversation persistence
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

print("✓ Graph compiled successfully with MemorySaver checkpoint")

## Graph Visualization
Display the workflow diagram

In [ ]:
# Display graph visualization
try:
    graph_image = graph.get_graph().draw_mermaid_png()
    display(Image(graph_image))
    print("✓ Graph visualization rendered")
except Exception as e:
    print(f"ℹ️ Graph visualization note: {e}")
    print("\nGraph structure:")
    print("  START → worker → [tools → worker]* → evaluator → [worker | END]")
    print("\nWhere:")
    print("  - worker: Processes tasks and decides whether to use tools")
    print("  - tools: Executes tool calls (browser, search, etc.)")
    print("  - evaluator: Assesses output quality and decides next step")

## Gradio UI with Streaming Support
Interactive interface for the Sidekick system

In [ ]:
def make_thread_id() -> str:
    """
    Generate unique thread ID for conversation tracking.
    Uses UUID v4 for uniqueness.
    
    Returns:
        str: Unique thread identifier
    """
    return str(uuid.uuid4())


async def process_message(
    message: str,
    success_criteria: str,
    history: list,
    thread: str
) -> list:
    """
    Process user message through the Sidekick graph with streaming support.
    Implements comprehensive exception handling and input/output guardrails.
    
    Args:
        message: User's request message
        success_criteria: Success criteria for the task
        history: Conversation history (list of message dicts)
        thread: Thread ID for state management and memory
        
    Returns:
        list: Updated conversation history with new messages
    """
    try:
        # Apply input guardrails to user message
        is_valid, processed_message = apply_guardrails_to_message(message, "input")
        if not is_valid:
            error_msg = {"role": "assistant", "content": f"⚠️ **Input Error:** {processed_message}"}
            return history + [error_msg]
        
        # Validate success criteria
        if not validate_success_criteria(success_criteria):
            error_msg = {
                "role": "assistant",
                "content": "⚠️ **Criteria Error:** Please provide clear success criteria (10-2000 characters)."
            }
            return history + [error_msg]
        
        # Configure thread for memory checkpoint
        config = {"configurable": {"thread_id": thread}}

        # Initialize graph state
        state = {
            "messages": [HumanMessage(content=processed_message)],
            "success_criteria": success_criteria,
            "feedback_on_work": None,
            "success_criteria_met": False,
            "user_input_needed": False,
            "notification_sent": False
        }
        
        # Invoke graph with async support (streaming via ainvoke)
        result = await graph.ainvoke(state, config=config)
        
        # Extract and format responses
        user_entry = {"role": "user", "content": processed_message}
        
        # Get assistant response (second to last message before evaluator feedback)
        if len(result["messages"]) >= 2:
            assistant_content = result["messages"][-2].content
            # Apply output guardrails
            is_valid, filtered_content = apply_guardrails_to_message(assistant_content, "output")
            reply = {"role": "assistant", "content": filtered_content}
        else:
            reply = {"role": "assistant", "content": "⚠️ No response generated. Please try again."}
        
        # Get evaluator feedback (last message)
        if len(result["messages"]) >= 1:
            feedback_content = result["messages"][-1].content
            feedback = {"role": "assistant", "content": f"📊 **Evaluator Feedback:** {feedback_content}"}
        else:
            feedback = {"role": "assistant", "content": "ℹ️ No feedback available."}
        
        return history + [user_entry, reply, feedback]
        
    except Exception as e:
        # Comprehensive error handling with traceback
        error_details = traceback.format_exc()
        error_msg = {
            "role": "assistant",
            "content": f"""⚠️ **System Error:** {str(e)}

**Details:**
```{error_details[:800]}```

Please try again or check your API keys and configuration."""
        }
        return history + [error_msg]


async def reset() -> tuple:
    """
    Reset the conversation state and generate new thread ID.
    
    Returns:
        tuple: (empty_message, empty_criteria, empty_chatbot, new_thread_id)
    """
    return "", "", None, make_thread_id()


print("✓ Gradio handler functions defined")

## Launch Gradio Interface
Create and launch the Sidekick UI with examples

In [ ]:
# Create Gradio interface with custom theme and styling
with gr.Blocks(
    theme=gr.themes.Default(primary_hue="emerald"),
    title="Sidekick Personal Co-worker",
    css="""
    .gradio-container { max-width: 900px !important; }
    .chatbot-container { border: 2px solid #10b981; border-radius: 10px; }
    """
) as demo:
    
    # Header
    gr.Markdown("# 🤖 Sidekick Personal Co-worker")
    gr.Markdown("*An intelligent multi-agent assistant powered by LangGraph, OpenAI, and Pushover Notifications*")
    
    # Thread state for conversation memory
    thread = gr.State(make_thread_id())
    
    # Chatbot display
    with gr.Row():
        chatbot = gr.Chatbot(
            label="Sidekick Conversation",
            height=450,
            type="messages",
            show_copy_button=True,
            placeholder="Your conversation with Sidekick will appear here..."
        )
    
    # Input fields
    with gr.Group():
        with gr.Row():
            message = gr.Textbox(
                label="Your Request",
                placeholder="Describe what you want your Sidekick to help you with (e.g., 'Research the latest AI trends and summarize key findings')...",
                lines=2,
                max_chars=5000
            )
        with gr.Row():
            success_criteria = gr.Textbox(
                label="Success Criteria",
                placeholder="Define what success looks like (e.g., 'Provide 3-5 key trends with specific examples from reputable sources, include links')...",
                lines=2,
                max_chars=2000
            )
    
    # Action buttons
    with gr.Row():
        reset_button = gr.Button("🔄 Reset Conversation", variant="stop")
        go_button = gr.Button("🚀 Go!", variant="primary", scale=2)
    
    # Event handlers
    message.submit(
        fn=process_message,
        inputs=[message, success_criteria, chatbot, thread],
        outputs=[chatbot]
    )
    success_criteria.submit(
        fn=process_message,
        inputs=[message, success_criteria, chatbot, thread],
        outputs=[chatbot]
    )
    go_button.click(
        fn=process_message,
        inputs=[message, success_criteria, chatbot, thread],
        outputs=[chatbot]
    )
    reset_button.click(
        fn=reset,
        inputs=[],
        outputs=[message, success_criteria, chatbot, thread]
    )
    
    # Example prompts
    gr.Examples(
        examples=[
            [
                "Research the latest developments in AI agents and summarize key trends from the past month",
                "Provide a concise summary of 3-5 key trends with specific examples, dates, and links to reputable sources"
            ],
            [
                "Find information about LangGraph's multi-agent capabilities and create a comparison with other frameworks",
                "Extract specific features, provide pros/cons, and include links to official documentation and GitHub repos"
            ],
            [
                "Search for the best Python libraries for data visualization and create a summary table",
                "List top 5 libraries with features, installation commands, use cases, and example code snippets"
            ]
        ],
        inputs=[message, success_criteria],
        label="Try these examples:"
    )
    
    # Footer
    gr.Markdown("""
    ### ℹ️ Usage Tips
    - **Be specific** in your request and success criteria for best results
    - Sidekick can use web search, Wikipedia, browser automation, and file operations
    - Push notifications will be sent when tasks complete (if Pushover is configured)
    - The evaluator ensures quality before marking tasks as complete
    
    **Author:** Samuel Kalu, Team Euclid | **Week:** 4 - LangGraph Multi-Agent Systems
    """)

print("✓ Gradio interface configured")

# Launch the interface with Colab-compatible settings
try:
    # For Colab: use share=True for public URL
    demo.launch(
        share=True,              # Public URL (works in Colab)
        server_name="0.0.0.0",   # Allow all connections
        server_port=7860,        # Default Gradio port
        show_error=True,         # Show detailed errors
        quiet=False,             # Show launch messages
        debug=True               # Enable debug mode for troubleshooting
    )
except Exception as e:
    print(f"⚠️ Gradio launch error: {e}")
    print("Trying alternative launch configuration...")
    # Fallback: try with different port
    try:
        demo.launch(
            share=True,
            server_port=7861,    # Try alternate port
            show_error=True
        )
    except Exception as fallback_error:
        print(f"❌ Failed to launch: {fallback_error}")
        print("Please check your Colab environment and try again.")

## Summary

### ✅ Features Implemented

**Core LangGraph Features:**
- ✅ **Structured Outputs**: Pydantic-based `EvaluatorOutput` schema for type-safe evaluation
- ✅ **Multi-Agent Flow**: Worker + Evaluator agents with intelligent orchestration
- ✅ **State Management**: TypedDict with message reduction for conversation tracking
- ✅ **Memory Checkpointing**: Thread-based persistence with `MemorySaver`

**Tools Integration:**
- ✅ **Browser Automation**: Playwright for web navigation and interaction
- ✅ **Web Search**: Google Serper API for current information
- ✅ **Wikipedia**: Factual knowledge retrieval
- ✅ **Python REPL**: Code execution capabilities
- ✅ **File Management**: Sandboxed file operations
- ✅ **Pushover Notifications**: Real-time push notifications to mobile devices

**Safety & Quality:**
- ✅ **Input Guardrails**: Text validation, sanitization, length checks, injection prevention
- ✅ **Output Guardrails**: Content validation, sensitive info filtering, redaction
- ✅ **Exception Handling**: Comprehensive try-catch blocks with full traceback logging

**Model Configuration:**
- ✅ **Sampling Parameters**: Temperature, max_tokens, streaming, timeout, retries
- ✅ **Model Selection**: gpt-4o-mini for both worker and evaluator
- ✅ **Streaming Support**: Async processing with `ainvoke`

**User Interface:**
- ✅ **Gradio UI**: Interactive interface with chatbot, examples, and reset
- ✅ **Conversation Memory**: Thread-based state management
- ✅ **Example Prompts**: Pre-configured examples for quick testing

**Colab Compatibility:**
- ✅ **nest_asyncio**: Async event loop compatibility
- ✅ **Headless Browser**: Chromium for Colab environment
- ✅ **Colab Secrets**: API key management via Settings → Secrets
- ✅ **Latest Versions**: All libraries use latest stable versions

### 🏗️ Architecture

```
START → Worker → [Tools → Worker]* → Evaluator → [Worker | END]
                ↑                    ↓
                └────────────────────┘
```

**Flow:**
1. **START** → Worker node receives user request
2. **Worker** processes task, decides to use tools or respond
3. **Tools** (if needed) execute and return results
4. **Worker** continues with tool results
5. **Evaluator** assesses output quality against criteria
6. **Decision**: Continue working OR end (success/user input needed)
7. **END** → Final response with evaluator feedback

### 📋 Usage Instructions

1. **Setup API Keys** (Colab Secrets):
   - `OPENAI_API_KEY`: Your OpenAI API key (required)
   - `PUSHOVER_TOKEN`: Pushover application token (optional)
   - `PUSHOVER_USER`: Pushover user key (optional)
   - `SERPER_API_KEY`: Google Serper API key for web search (optional)

2. **Run the Notebook**:
   - Execute all cells from top to bottom
   - Wait for Gradio interface to launch
   - Interface will be available at `http://localhost:7860`

3. **Interact with Sidekick**:
   - Enter your request in the "Your Request" field
   - Define success criteria (what a good answer looks like)
   - Click "Go!" to start the multi-agent workflow
   - Watch the conversation unfold with evaluator feedback

4. **Pushover Notifications**:
   - Configure Pushover credentials in Colab Secrets
   - Notifications sent automatically when tasks complete
   - Use the `send_push_notification` tool in your workflows

### 🎓 Key Learnings

- **LangGraph** provides excellent orchestration for multi-agent systems with clear state management
- **Structured Outputs** ensure consistent, type-safe evaluation behavior
- **Guardrails** are essential for production-ready AI systems (input validation, output filtering)
- **Streaming and Async** patterns significantly improve user experience
- **Exception Handling** with detailed tracebacks makes debugging much easier
- **Pushover Integration** demonstrates how to add external notification services
- **Colab Compatibility** requires specific patterns (nest_asyncio, headless browsers, secrets)

### 🚀 Future Enhancements

- Add support for multiple LLM providers (Anthropic, Google, etc.)
- Implement conversation export (PDF, Markdown)
- Add voice input/output capabilities
- Integrate with Slack/Discord for team notifications
- Add advanced planning and reasoning capabilities

---

**Author:** Samuel Kalu  
**Team:** Euclid  
**Bootcamp:** LLM Engineering  
**Week:** 4 - LangGraph Multi-Agent Systems  
**Date:** March 2026